In [ ]:
MODE = 'PDE' # Options: 'PDE', 'PFE'

In [ ]:
import pandas as pd
import numpy as np

if MODE == 'PDE':
    df = pd.read_csv('output/pde/catalogue/events_filtered.csv')
elif MODE == 'PFE':
    df = pd.read_csv('output/pfe/catalogue/events_filtered.csv')

In [ ]:
def get_metrics(df, target_team, window_sec = 300, event_type=None, is_post_window=False):
        
        if df.empty or window_sec <= 0:
            return {m: 0.0 for m in ['pass_acc_ratio', 'fwd_pass_ratio', 'long_ball_ratio', 'xg_rate', 
                                    'shot_rate', 'avg_xg', 'dribble_rate', 'turnover_rate', 
                                    'foul_rate', 'press_intensity']}
        
        team_df = df[df.team == target_team].copy()
        
        # Filter out non-open play events for passing and shooting metrics
        if 'pass_type' in team_df.columns:
            team_df = team_df[~team_df.pass_type.isin(['Corner', 'Free Kick', 'Goal Kick', 'Throw-in'])]
        if 'shot_type' in team_df.columns:
            team_df = team_df[~team_df.shot_type.isin(['Penalty', 'Free Kick'])]

        res = {}
        
        # Passing Metrics
        passes = team_df[team_df.type == 'Pass']
        n_passes = len(passes)
        if n_passes > 0:
            # Pass completion rate: Successful / Total
            res['pass_acc_ratio'] = len(passes[passes.pass_outcome.isnull()]) / n_passes
            
            # Forward pass ratio: Forward / Total
            res['fwd_pass_ratio'] = (passes['end_x'] > passes['x']).sum() / n_passes
            
            # Long ball ratio: Long (>32m) / Total
            distances = np.sqrt((passes['end_x'] - passes['x'])**2 + (passes['end_y'] - passes['y'])**2)
            res['long_ball_ratio'] = (distances > 32).sum() / n_passes
        else:
            res['pass_acc_ratio'] = res['fwd_pass_ratio'] = res['long_ball_ratio'] = 0.0

        # Shooting Metrics
        shots = team_df[team_df.type == 'Shot']
        if 'total_seconds' in shots.columns:
            shots = shots.sort_values('total_seconds')

        if is_post_window and event_type in {'Converted Turnover', 'Final Third Dribble'} and len(shots) > 0:
            shots = shots.iloc[1:]

        res['xg_rate'] = shots.shot_statsbomb_xg.sum() if len(shots) > 0 else 0.0
        res['shot_rate'] = len(shots)
        res['avg_xg'] = shots.shot_statsbomb_xg.mean() if len(shots) > 0 else 0.0

        # Dribbling and Turnover Metrics
        res['dribble_rate'] = len(team_df[team_df.type == 'Dribble'])
        res['turnover_rate'] = len(team_df[team_df.type.isin(['Dispossessed', 'Miscontrol'])])

        # Defensive Metrics
        res['foul_rate'] = len(team_df[team_df.type == 'Foul Committed'])
        res['press_intensity'] = len(team_df[team_df.type == 'Pressure'])
        
        return res

In [ ]:
def calculate_event_behavioral_response_metrics(event_row, events, home_team_name, W_seconds=300):
    
    t = event_row['total_seconds']
    team = event_row['affected_team']
    period = event_row['period']

    # Define pre and post windows considering half-time boundaries
    pre_events = events[(events.total_seconds >= t - W_seconds) & 
                        (events.total_seconds < t) & 
                        (events.period == period)]
    
    post_events = events[(events.total_seconds > t) & 
                         (events.total_seconds <= t + W_seconds) & 
                         (events.period == period)]

    # Calculate metrics for both windows
    event_type = event_row.get('event_type', None)
    pre_m = get_metrics(pre_events, team, event_type=event_type, is_post_window=False)
    post_m = get_metrics(post_events, team, event_type=event_type, is_post_window=True)

    # Combine into a single row
    output = event_row.to_dict()
    for metric in pre_m.keys():
        output[f'pre_{metric}'] = pre_m[metric]
        output[f'post_{metric}'] = post_m[metric]
        output[f'delta_{metric}'] = post_m[metric] - pre_m[metric]
    
    # Home/Away indicator
    output['is_home'] = 1 if team == home_team_name else 0

    # Compute red card difference in the game
    events_before_t = events[events.total_seconds <= t]
    focal_reds = 0
    opp_reds = 0
    
    if 'foul_committed_card' in events.columns:
        reds = events_before_t[events_before_t['foul_committed_card'].isin(['Red Card', 'Second Yellow'])]
        focal_reds += len(reds[reds.team == team])
        opp_reds += len(reds[reds.team != team])
        
    if 'bad_behaviour_card' in events.columns:
        reds_bb = events_before_t[events_before_t['bad_behaviour_card'].isin(['Red Card', 'Second Yellow'])]
        focal_reds += len(reds_bb[reds_bb.team == team])
        opp_reds += len(reds_bb[reds_bb.team != team])
        
    output['red_card_diff'] = focal_reds - opp_reds

    # y_conceded: Check if the affected team concedes a goal in the next 10 minutes
    events_next_10m = events[(events.total_seconds > t) & (events.total_seconds <= t + 600)]
    
    target_met = 0
    if 'shot_outcome' in events_next_10m.columns:
        if MODE == 'PDE':
            # PDE: Did the team concede a goal? (The opposing team scored a goal)
            goals = events_next_10m[(events_next_10m.type == 'Shot') & 
                                    (events_next_10m.shot_outcome == 'Goal') & 
                                    (events_next_10m.team != team)]
        elif MODE == 'PFE':
            # PFE: Did the team score a goal? (The examined team scored a goal)
            goals = events_next_10m[(events_next_10m.type == 'Shot') & 
                                    (events_next_10m.shot_outcome == 'Goal') & 
                                    (events_next_10m.team == team)]
            
        if len(goals) > 0:
            target_met = 1

    output['y'] = target_met

    return output

In [ ]:
import os
import glob

# Read all standard match files to extract home team names
files = glob.glob("sb_offline_data/match_*.pkl")

home_teams_dict = {}
match_ids = []
missing = 0

for file in files:
    # Get match ID from filename
    m_id = int(os.path.basename(file).split('_')[1].split('.')[0])
    match_ids.append(m_id)
    
    # Load events
    events = pd.read_pickle(file)

    # Extract the home team from the Starting XI events
    starting_xis = events[events['type'] == 'Starting XI']
    if not starting_xis.empty:
        home_team = starting_xis.iloc[0]['team']
        home_teams_dict[m_id] = home_team
    else:
        home_teams_dict[m_id] = "Unknown"
        missing += 1

In [6]:
print(f"Processed {len(match_ids)} matches. Missing home team info for {missing} matches.")

Processed 1238 matches. Missing home team info for 0 matches.


In [ ]:
final_behavioral_data = []

for m_id, match_events in df.groupby('match_id'):
    
    current_home_team = home_teams_dict.get(m_id, "Unknown")
    events = pd.read_pickle(f"sb_offline_data/match_{m_id}.pkl")
    
    events['total_seconds'] = events['minute'] * 60 + events['second']

    events['x'] = events['location'].str[0]
    events['y'] = events['location'].str[1]
    
    if 'pass_end_location' in events.columns:
        events['end_x'] = events['pass_end_location'].str[0]
        events['end_y'] = events['pass_end_location'].str[1]
    else:
        events['end_x'] = np.nan
        events['end_y'] = np.nan
    
    for _, event_row in match_events.iterrows():
        rich_event = calculate_event_behavioral_response_metrics(event_row, events, current_home_team)
        final_behavioral_data.append(rich_event)

master_df = pd.DataFrame(final_behavioral_data)

if MODE == 'PDE':
    master_df.to_csv('output/pde/catalogue/events_with_metrics.csv', index=False, encoding='utf-8-sig')
elif MODE == 'PFE':
    master_df.to_csv('output/pfe/catalogue/events_with_metrics.csv', index=False, encoding='utf-8-sig')